# 02 — Full Experiment Pipeline
> BTK Datathon 2026 | Metric: **MSE** on `career_success_score` (0–100)  
> Preprocessing → FE → TF-IDF OOF → LightGBM + XGBoost + CatBoost → Ensemble

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import sparse
from scipy.optimize import minimize
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
import xgboost as xgb

try:
    import catboost as cb
    HAS_CB = True
    print('CatBoost OK')
except ImportError:
    HAS_CB = False
    print('CatBoost not installed — skipping')

SEED = 42
np.random.seed(SEED)
print('Imports OK')

In [ ]:
if Path('/kaggle/input').exists():
    DATA = next(Path('/kaggle/input').iterdir())
else:
    DATA = Path('../datathon-2026')
SUBDIR = Path('../submissions')

train = pd.read_csv(DATA / 'train.csv')
test  = pd.read_csv(DATA / 'test_x.csv')
y     = train['career_success_score'].values
kf    = KFold(n_splits=5, shuffle=True, random_state=SEED)

print(f'train: {train.shape} | test: {test.shape}')
print(f'y: mean={y.mean():.2f} std={y.std():.2f} min={y.min():.1f} max={y.max():.1f}')

In [ ]:
# Tier 1 = best university → highest ordinal value
TIER_MAP = {'Tier 1': 4, 'Tier 2': 3, 'Tier 3': 2, 'Tier 4': 1}

MISS_COLS = [
    'internship_duration_months', 'english_exam_score', 'github_avg_stars',
    'open_source_contribution_count', 'hr_interview_score',
    'linkedin_profile_score', 'portfolio_score',
]
CAT_COLS   = ['department', 'target_role', 'hobby', 'preferred_social_media_platform']
DROP_TRAIN = ['student_id', 'career_success_score', 'mentor_feedback_text']
DROP_TEST  = ['student_id', 'mentor_feedback_text']

TECH_COLS = [
    'coding_score', 'problem_solving_score', 'data_structures_score', 'sql_score',
    'machine_learning_score', 'backend_score', 'frontend_score', 'cloud_score', 'devops_score',
]
SOFT_COLS = ['communication_score', 'teamwork_score', 'leadership_score', 'presentation_score']

In [ ]:
def preprocess(df):
    df = df.copy()
    # Missing-value indicator flags (row-level, no leakage)
    for c in MISS_COLS:
        df[f'{c}_isna'] = df[c].isna().astype(np.int8)
    # Domain rule: no internship => duration = 0
    df.loc[df['internship_count'] == 0, 'internship_duration_months'] = 0.0
    # Ordinal encoding for university tier
    df['university_tier_ord'] = df['university_tier'].map(TIER_MAP)
    # Time-derived features
    df['years_after_grad'] = df['application_year'] - df['graduation_year']
    df['age_at_grad']      = df['age'] - df['years_after_grad']
    return df

In [ ]:
def add_features(df):
    df = df.copy()

    # --- Efficiency / conversion ratios
    df['interview_rate']       = df['interviews_attended'] / (df['applications_sent'] + 1)
    df['award_per_hackathon']  = df['hackathon_awards']    / (df['hackathon_count'] + 1)
    df['internship_intensity'] = df['internship_duration_months'].fillna(0) / (df['internship_count'] + 1)

    # --- GitHub signal (log-compressed to reduce outlier effect)
    df['stars_total']  = df['github_avg_stars'].fillna(0) * df['github_repo_count']
    df['github_score'] = (
        np.log1p(df['stars_total'])
        + np.log1p(df['open_source_contribution_count'].fillna(0))
        + np.log1p(df['github_repo_count'])
    )

    # --- Technical skills block
    df['tech_mean']  = df[TECH_COLS].mean(axis=1)
    df['tech_max']   = df[TECH_COLS].max(axis=1)
    df['tech_min']   = df[TECH_COLS].min(axis=1)
    df['tech_std']   = df[TECH_COLS].std(axis=1)
    df['tech_range'] = df['tech_max'] - df['tech_min']
    df['tech_sum']   = df[TECH_COLS].sum(axis=1)

    # --- Soft skills block
    df['soft_mean'] = df[SOFT_COLS].mean(axis=1)
    df['soft_max']  = df[SOFT_COLS].max(axis=1)
    df['soft_std']  = df[SOFT_COLS].std(axis=1)

    # --- Interview composite
    hr_med = df['hr_interview_score'].median()
    df['interview_composite'] = (
        df['technical_interview_score'] + df['hr_interview_score'].fillna(hr_med)
    ) / 2

    # --- Experience breadth
    df['experience_total'] = (
        df['internship_count'] + df['freelance_project_count']
        + df['real_client_project_count'] + df['hackathon_count']
    )
    df['cert_density'] = df['certification_count'] / (df['years_after_grad'] + 1)

    # --- Composite strength score
    port_med = df['portfolio_score'].median()
    df['profile_strength'] = (
        df['tech_mean'] + df['soft_mean']
        + df['portfolio_score'].fillna(port_med) + df['cv_quality_score']
    ) / 4

    # --- Interaction / cross features
    df['tech_x_interview']  = df['tech_mean']  * df['technical_interview_score']               / 100
    df['soft_x_hr']         = df['soft_mean']  * df['hr_interview_score'].fillna(hr_med)        / 100
    df['cgpa_x_tech']       = df['cgpa']        * df['tech_mean']                               / 10
    df['project_x_tech']    = df['project_quality_score'] * df['tech_mean']                     / 100
    df['cgpa_x_attendance'] = df['cgpa']        * df['attendance_rate']                         / 100

    return df


POSITIVE_KW = [
    'ba\u015far\u0131l\u0131', 'm\u00fckemmel', 'geli\u015fmi\u015f', 'g\u00fc\u00e7l\u00fc',
    'umut verici', 'dikkat \u00e7ekici', 'yetenekli', 'lider', '\u00fcst\u00fcn', 'harika',
    'olumlu', 'etkileyici', '\u00f6zveri', 'potansiyel', 'yarat\u0131c\u0131',
    'ba\u015far\u0131', 'liderlik', 'g\u00fcven', 'de\u011fer', 'motive',
]
NEGATIVE_KW = [
    'geli\u015fmeli', 'eksik', 'zay\u0131f', 'yetersiz', 'dikkat', 'sorun',
    '\u00e7al\u0131\u015fmas\u0131 gerekiyor', 'd\u00fc\u015f\u00fck', 'ba\u015far\u0131s\u0131z',
    'olumsuz', 'endi\u015fe', 'risk', 'ihtiya\u00e7', 'a\u015fikar',
    'eksiklik', 'iyile\u015ftirmeli', 'kritik', 'acil', 's\u0131n\u0131rl\u0131',
]


def add_text_stats(df):
    df   = df.copy()
    txt  = df['mentor_feedback_text'].fillna('')
    tlow = txt.str.lower()

    df['text_len']          = txt.str.len()
    df['text_word_count']   = txt.str.split().str.len()
    df['text_avg_word_len'] = df['text_len'] / (df['text_word_count'] + 1)
    df['sentence_count']    = txt.str.count(r'[.!?]+') + 1

    df['positive_count']  = tlow.apply(lambda x: sum(x.count(w) for w in POSITIVE_KW))
    df['negative_count']  = tlow.apply(lambda x: sum(x.count(w) for w in NEGATIVE_KW))
    df['sentiment_ratio'] = df['positive_count'] / (df['negative_count'] + 1)
    df['sentiment_diff']  = df['positive_count'] - df['negative_count']
    df['sentiment_score'] = df['sentiment_diff'] / (df['text_word_count'] + 1)

    return df


def build_all_features(df):
    return add_text_stats(add_features(preprocess(df)))


print('Feature functions defined')

In [ ]:
train_f = build_all_features(train)
test_f  = build_all_features(test)

X      = train_f.drop(columns=DROP_TRAIN)
X_test = test_f.drop(columns=DROP_TEST)

train_texts = train['mentor_feedback_text'].fillna('').tolist()
test_texts  = test['mentor_feedback_text'].fillna('').tolist()

print(f'X={X.shape}  X_test={X_test.shape}')
print(f'Missing in X: {X.isna().sum().sum()} cells')

## TF-IDF OOF Feature
Word (1–3 gram) + char_wb (3–5 gram) TF-IDF fed into Ridge.  
The OOF prediction is fold-safe: each val row was predicted using a model trained on the other 4 folds only.  
We then inject this OOF column into X so all downstream tree models benefit from the text signal.

In [ ]:
def make_tfidf_oof(train_texts, test_texts, y, kf, alpha=10.0):
    oof       = np.zeros(len(train_texts))
    test_pred = np.zeros(len(test_texts))

    for tr_idx, val_idx in kf.split(range(len(train_texts))):
        tr_txt  = [train_texts[i] for i in tr_idx]
        val_txt = [train_texts[i] for i in val_idx]

        vec_w = TfidfVectorizer(analyzer='word',    ngram_range=(1, 3),
                                max_features=15000, sublinear_tf=True, min_df=2)
        vec_c = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5),
                                max_features=15000, sublinear_tf=True, min_df=3)

        Xtr  = sparse.hstack([vec_w.fit_transform(tr_txt),  vec_c.fit_transform(tr_txt)])
        Xval = sparse.hstack([vec_w.transform(val_txt),     vec_c.transform(val_txt)])
        Xte  = sparse.hstack([vec_w.transform(test_texts),  vec_c.transform(test_texts)])

        ridge = Ridge(alpha=alpha)
        ridge.fit(Xtr, y[tr_idx])
        oof[val_idx]  = ridge.predict(Xval)
        test_pred    += ridge.predict(Xte) / kf.n_splits

    mse = ((y - np.clip(oof, 0, 100)) ** 2).mean()
    print(f'TF-IDF Ridge standalone CV MSE: {mse:.4f}')
    return oof, test_pred


tfidf_oof, tfidf_test = make_tfidf_oof(train_texts, test_texts, y, kf)

## (Optional) Multilingual Sentence Embeddings
Requires `pip install sentence-transformers`.  
Uses `paraphrase-multilingual-MiniLM-L12-v2` — handles Turkish natively.  
Embeddings are reduced to 64 dims with TruncatedSVD and appended to X.

In [ ]:
HAS_SBERT = False
try:
    from sentence_transformers import SentenceTransformer
    from sklearn.decomposition import TruncatedSVD

    print('Loading multilingual sentence transformer ...')
    sbert     = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
    all_texts = train_texts + test_texts
    emb       = sbert.encode(all_texts, batch_size=64, show_progress_bar=True,
                              normalize_embeddings=True)

    svd   = TruncatedSVD(n_components=64, random_state=SEED)
    emb_r = svd.fit_transform(emb)

    emb_cols     = [f'sbert_{i}' for i in range(64)]
    train_emb_df = pd.DataFrame(emb_r[:len(train_texts)], columns=emb_cols)
    test_emb_df  = pd.DataFrame(emb_r[len(train_texts):], columns=emb_cols)
    HAS_SBERT    = True
    print(f'Embeddings OK: {emb_r.shape}  explained_var={svd.explained_variance_ratio_.sum():.3f}')
except Exception as e:
    print(f'sentence-transformers not available — skipping ({e})')

In [ ]:
# Inject TF-IDF OOF as a numeric feature (fold-safe: same kf splits used)
X['tfidf_oof']      = tfidf_oof
X_test['tfidf_oof'] = tfidf_test

if HAS_SBERT:
    X      = pd.concat([X.reset_index(drop=True),      train_emb_df], axis=1)
    X_test = pd.concat([X_test.reset_index(drop=True), test_emb_df],  axis=1)

print(f'Final feature matrix: X={X.shape}  X_test={X_test.shape}')

## Model 1 — LightGBM

In [ ]:
LGBM_PARAMS = dict(
    objective='regression', metric='mse',
    n_estimators=5000, learning_rate=0.02,
    num_leaves=127, max_depth=-1,
    min_child_samples=15,
    subsample=0.8, subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.05, reg_lambda=0.5,
    random_state=SEED, verbose=-1,
)


def run_lgbm(X, y, X_test, cat_cols=CAT_COLS, params=None):
    params = params or LGBM_PARAMS
    X, X_test = X.copy(), X_test.copy()
    present = [c for c in cat_cols if c in X.columns]
    for c in present:
        X[c]      = X[c].astype('category')
        X_test[c] = pd.Categorical(X_test[c], categories=X[c].cat.categories)

    oof = np.zeros(len(X)); test_pred = np.zeros(len(X_test))
    for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
        m = lgb.LGBMRegressor(**params)
        m.fit(
            X.iloc[tr_idx], y[tr_idx],
            eval_set=[(X.iloc[val_idx], y[val_idx])],
            eval_metric='mse',
            callbacks=[lgb.early_stopping(200, verbose=False), lgb.log_evaluation(0)],
        )
        oof[val_idx]  = m.predict(X.iloc[val_idx])
        test_pred    += m.predict(X_test) / kf.n_splits
        fold_mse = ((y[val_idx] - np.clip(oof[val_idx], 0, 100)) ** 2).mean()
        print(f'  fold {fold+1}: MSE={fold_mse:.4f}  iter={m.best_iteration_}')

    oof, test_pred = np.clip(oof, 0, 100), np.clip(test_pred, 0, 100)
    cv = ((y - oof) ** 2).mean()
    print(f'LightGBM CV MSE: {cv:.4f}')
    return oof, test_pred, cv


lgbm_oof, lgbm_test, lgbm_cv = run_lgbm(X, y, X_test)

## Model 2 — XGBoost

In [ ]:
def run_xgb(X, y, X_test, cat_cols=CAT_COLS):
    X, X_test = X.copy(), X_test.copy()
    present = [c for c in cat_cols if c in X.columns]
    for c in present:
        le = LabelEncoder().fit(pd.concat([X[c], X_test[c]]).astype(str))
        X[c]      = le.transform(X[c].astype(str))
        X_test[c] = le.transform(X_test[c].astype(str))

    oof = np.zeros(len(X)); test_pred = np.zeros(len(X_test))
    for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
        m = xgb.XGBRegressor(
            n_estimators=5000, learning_rate=0.02,
            max_depth=7, subsample=0.8, colsample_bytree=0.8,
            min_child_weight=3, reg_alpha=0.05, reg_lambda=0.5,
            objective='reg:squarederror', tree_method='hist',
            random_state=SEED, verbosity=0, early_stopping_rounds=200,
        )
        m.fit(
            X.iloc[tr_idx], y[tr_idx],
            eval_set=[(X.iloc[val_idx], y[val_idx])],
            verbose=False,
        )
        oof[val_idx]  = m.predict(X.iloc[val_idx])
        test_pred    += m.predict(X_test) / kf.n_splits
        fold_mse = ((y[val_idx] - np.clip(oof[val_idx], 0, 100)) ** 2).mean()
        print(f'  fold {fold+1}: MSE={fold_mse:.4f}')

    oof, test_pred = np.clip(oof, 0, 100), np.clip(test_pred, 0, 100)
    cv = ((y - oof) ** 2).mean()
    print(f'XGBoost CV MSE: {cv:.4f}')
    return oof, test_pred, cv


xgb_oof, xgb_test, xgb_cv = run_xgb(X, y, X_test)

## Model 3 — CatBoost  
Native categorical support — no label encoding needed.

In [ ]:
if HAS_CB:
    def run_catboost(X, y, X_test, cat_cols=CAT_COLS):
        X, X_test = X.copy(), X_test.copy()
        present = [c for c in cat_cols if c in X.columns]
        for c in present:
            X[c]      = X[c].astype(str).fillna('NA')
            X_test[c] = X_test[c].astype(str).fillna('NA')
        cat_idx = [list(X.columns).index(c) for c in present]

        oof = np.zeros(len(X)); test_pred = np.zeros(len(X_test))
        for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
            tr_pool  = cb.Pool(X.iloc[tr_idx],  y[tr_idx],  cat_features=cat_idx)
            val_pool = cb.Pool(X.iloc[val_idx], y[val_idx], cat_features=cat_idx)
            te_pool  = cb.Pool(X_test,                       cat_features=cat_idx)

            m = cb.CatBoostRegressor(
                iterations=4000, learning_rate=0.03,
                depth=7, l2_leaf_reg=3,
                loss_function='RMSE', eval_metric='RMSE',
                random_seed=SEED, verbose=0, early_stopping_rounds=200,
            )
            m.fit(tr_pool, eval_set=val_pool, use_best_model=True)
            oof[val_idx]  = m.predict(val_pool)
            test_pred    += m.predict(te_pool) / kf.n_splits
            fold_mse = ((y[val_idx] - np.clip(oof[val_idx], 0, 100)) ** 2).mean()
            print(f'  fold {fold+1}: MSE={fold_mse:.4f}')

        oof, test_pred = np.clip(oof, 0, 100), np.clip(test_pred, 0, 100)
        cv = ((y - oof) ** 2).mean()
        print(f'CatBoost CV MSE: {cv:.4f}')
        return oof, test_pred, cv

    cb_oof, cb_test, cb_cv = run_catboost(X, y, X_test)
else:
    cb_oof = cb_test = None
    cb_cv  = float('inf')
    print('CatBoost skipped')

## Feature Importance (LightGBM)

In [ ]:
X_imp = X.copy()
for c in [c for c in CAT_COLS if c in X_imp.columns]:
    X_imp[c] = X_imp[c].astype('category')

m_imp = lgb.LGBMRegressor(**{**LGBM_PARAMS, 'n_estimators': 600, 'learning_rate': 0.05})
m_imp.fit(X_imp, y)

imp = pd.Series(m_imp.feature_importances_, index=X_imp.columns).sort_values(ascending=False)
print('Top 30 features:')
print(imp.head(30).to_string())
print(f'\nBottom 10 (candidates for removal):')
print(imp.tail(10).to_string())

## Ensemble — CV-Optimal Blend
SLSQP optimization finds the weight vector `w` (summing to 1, all ≥ 0) that minimises OOF MSE.  
Multiple starting points guard against local minima.

In [ ]:
def optimize_blend(oofs_list, test_preds_list, y, names):
    stacked = np.column_stack(oofs_list)

    def loss(w):
        return ((y - np.clip((stacked * w).sum(axis=1), 0, 100)) ** 2).mean()

    n = len(oofs_list)
    constraints = [{'type': 'eq', 'fun': lambda w: w.sum() - 1}]
    bounds = [(0, 1)] * n

    best = None
    for w0 in [np.ones(n) / n] + [np.eye(n)[i] for i in range(n)]:
        res = minimize(loss, w0, method='SLSQP', constraints=constraints, bounds=bounds,
                       options={'ftol': 1e-10, 'maxiter': 3000})
        if best is None or res.fun < best.fun:
            best = res

    w = best.x
    print('Optimal blend weights:')
    for name, wi in zip(names, w):
        print(f'  {name:<25s}: {wi:.4f}')
    print(f'Ensemble CV MSE: {best.fun:.4f}')

    test_blend = np.clip(
        (np.column_stack(test_preds_list) * w).sum(axis=1), 0, 100
    )
    return w, best.fun, test_blend


all_oofs       = [lgbm_oof, xgb_oof, tfidf_oof]
all_test_preds = [lgbm_test, xgb_test, tfidf_test]
model_names    = ['LightGBM', 'XGBoost', 'TF-IDF Ridge']

if cb_oof is not None:
    all_oofs.append(cb_oof)
    all_test_preds.append(cb_test)
    model_names.append('CatBoost')

weights, ensemble_cv, ensemble_test = optimize_blend(all_oofs, all_test_preds, y, model_names)

# Summary table
print(f"\n{'Model':<25s}  CV MSE")
print('-' * 38)
for name, oof in zip(model_names, all_oofs):
    mse = ((y - np.clip(oof, 0, 100)) ** 2).mean()
    print(f'{name:<25s}  {mse:.4f}')
print(f"{'Ensemble':<25s}  {ensemble_cv:.4f}")

## (Optional) Optuna Hyperparameter Search — LightGBM
Run only if you have 30–60 min of spare compute. 100 trials is usually enough to squeeze out 1–2 MSE.

In [ ]:
RUN_OPTUNA = False  # Set True to run

if RUN_OPTUNA:
    try:
        import optuna
        optuna.logging.set_verbosity(optuna.logging.WARNING)

        def objective(trial):
            params = dict(
                objective='regression', metric='mse', verbose=-1,
                random_state=SEED,
                n_estimators=trial.suggest_int('n_estimators', 1000, 6000),
                learning_rate=trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
                num_leaves=trial.suggest_int('num_leaves', 31, 255),
                min_child_samples=trial.suggest_int('min_child_samples', 5, 50),
                subsample=trial.suggest_float('subsample', 0.5, 1.0),
                colsample_bytree=trial.suggest_float('colsample_bytree', 0.5, 1.0),
                reg_alpha=trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
                reg_lambda=trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
            )
            _, _, cv = run_lgbm(X, y, X_test, params=params)
            return cv

        study = optuna.create_study(
            direction='minimize',
            sampler=optuna.samplers.TPESampler(seed=SEED),
        )
        study.optimize(objective, n_trials=100, show_progress_bar=True)

        print('Best params:', study.best_params)
        print('Best CV MSE:', study.best_value)

        lgbm_opt_oof, lgbm_opt_test, lgbm_opt_cv = run_lgbm(
            X, y, X_test, params=study.best_params
        )
    except ImportError:
        print('optuna not installed — pip install optuna')

## Submission
**Sub 1:** Ensemble blend (best CV)  
**Sub 2:** LightGBM alone (diversity hedge — structurally different from ensemble)

In [ ]:
sample = pd.read_csv(DATA / 'sample_submission.csv')

def make_sub(preds, fname):
    sub = pd.DataFrame({
        'student_id': test['student_id'],
        'career_success_score': preds,
    })
    assert list(sub.columns) == list(sample.columns), 'Column mismatch!'
    assert len(sub) == len(test), 'Row count mismatch!'
    assert sub['student_id'].is_unique, 'Duplicate IDs!'
    assert sub['career_success_score'].notna().all(), 'NaN predictions!'
    sub.to_csv(SUBDIR / fname, index=False)
    print(f'Saved: {fname}  |  min={preds.min():.2f}  max={preds.max():.2f}  mean={preds.mean():.2f}')
    return sub


sub1 = make_sub(ensemble_test, f'sub_002_ensemble_cv{ensemble_cv:.2f}.csv')
sub2 = make_sub(lgbm_test,     f'sub_003_lgbm_cv{lgbm_cv:.2f}.csv')

print(f'\nFinal scores:')
print(f'  Ensemble CV MSE : {ensemble_cv:.4f}')
print(f'  LightGBM CV MSE : {lgbm_cv:.4f}')
print(f'  Baseline (01_nb): 83.83')